# Five repeated stratified splits: MLP embeddings, TabNet and FT-Transformer

**Class-weight experiment: CW0** — Class weighting disabled. All training samples/classes contribute with unit weight; 0 labels this experiment and is not a literal zero loss weight.

Five **repeated stratified holdouts with seeds 42, 123, 2026, 3407, 777**, using
**64/16/20** train/validation/test ratios as the ML notebook. Each partition must contain Severity 1–4. This is not five-fold CV. Use the same full input file and processed-feature schema
for matching rows. Optionally set `ML_RESULTS_DIR` to check every split hash against the ML results.

1. Fit numerical imputation/scaling and categorical vocabularies on training rows only.
2. Train all three architectures with **unweighted cross-entropy**. Class weighting disabled. All training samples/classes contribute with unit weight; 0 labels this experiment and is not a literal zero loss weight. TabNet uses `loss_fn`; `weights=0` disables weighted sampling.
3. Early stop on validation macro-F1 and restore the best model. Validation/test do not update embeddings.
4. Select independent one-vs-rest F1 thresholds using validation probabilities, preserving the ML rule:
   highest original probability among passing classes; training-majority fallback if none pass.
   This is not joint multiclass macro-F1 optimization.
5. Evaluate argmax and frozen tuned decisions on the same test probabilities. Never retune on test results.

MLP and FT-Transformer can use both T4 GPUs through optional `DataParallel`; TabNet uses GPU 0.
Models train sequentially. Two GPUs do not automatically combine their memory. The settings below are
conservative starting configurations, not tuned values or a guarantee that the full dataset will fit.
This version runs all five seeds. Reduce batches if GPU memory is insufficient.

Categorical IDs: 0=unknown, 1=missing, 2=rare training category; known retained categories start at 3.
The same training vocabulary feeds all models; their embeddings are learned independently.
Training-only rare-category grouping limits large Street/City/Zipcode vocabularies. Numeric/Boolean
features are supplied as scaled numbers. The 41 selected processed features are retained in their original order in both notebooks.
`End_Time` and `Distance(mi)` are excluded from model features.

Outputs: loss and macro-F1 curves, before/after metrics and reports, confusion-matrix CSVs/PNGs,
test probabilities, thresholds, split manifests, preprocessing, model checkpoints and a results ZIP.
Optional SHAP is disabled initially because it adds significant runtime.

Before training, all five splits are checked for nonoverlapping partitions and the presence of
Severity 1, 2, 3 and 4 in train, validation and test. Per-run class counts are saved in
`split_severity_counts.csv`. A dataset missing a severity class is rejected.

Per-severity tables include Accuracy, Precision, Recall, F1 and Support before and after
threshold tuning. Accuracy is binary one-vs-rest (TP + TN) / N for each severity;
Recall is TP / (TP + FN). Five-run summaries and paired changes include all four metrics.


## Processed dataset schema
This version requires 41 predictors (excluding Distance(mi)) from your supplied CSV header. It does not recreate or discard your time features.
Fourteen categorical columns, including Zipcode and Season, are explicitly declared; the remaining 27 are numerical, including nine Boolean 0/1 indicators.
CSV categorical dtypes preserve leading-zero ZIP codes. All preprocessing statistics and vocabularies are fitted on training rows only.
Outputs include feature_schema.csv and feature_schema.json for auditing types, missing values, ranges and feature order.
The time/cyclic features must have been created consistently upstream; this code preserves their supplied values.


In [ ]:
# Kaggle already provides CUDA-enabled PyTorch. Do not replace it with a CPU build.
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pytorch-tabnet>=4.1,<5', 'rtdl_revisiting_models>=0.0.2,<0.1',
    'scikit-learn>=1.4,<1.8', 'pandas', 'numpy', 'scipy', 'joblib', 'matplotlib'], check=True)

In [ ]:
import os
import json
import random
import gc
import hashlib
import time
import zipfile
import shutil
import importlib.metadata
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_curve, classification_report, confusion_matrix)

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.metrics import Metric
from rtdl_revisiting_models import FTTransformer

DATA_PATH = Path('/workspace/US_Accidents_processed_full.zip')  # Adjust if your dataset filename differs.
CSV_MEMBER = None
COMBINE_EXISTING_SPLITS = False
ML_RESULTS_DIR = None  # Optional Path('/kaggle/input/.../ml_repeated_splits_...')
SEEDS = [42, 123, 2026, 3407, 777]  # Five repeated stratified train/validation/test splits.
TARGET = 'Severity'
REQUIRED_SEVERITY_LEVELS = [1, 2, 3, 4]
MODEL_NAMES = ['MLP_Embeddings', 'TabNet', 'FTTransformer']
METRICS = ['Accuracy', 'Macro_Precision', 'Macro_Recall', 'Macro_F1']
USE_CLASS_WEIGHT = False
CLASS_WEIGHT_CAP = None  # No cap; weighting enabled only when USE_CLASS_WEIGHT=True.
CLASS_WEIGHT_MODE = "CW0"
THRESHOLD_BETA = 1.0
MIN_CATEGORY_COUNT = 5
MAX_CATEGORIES_PER_COLUMN = 10_000  # Excludes reserved IDs; increase only after checking RAM.
MAX_EPOCHS = 50
PATIENCE = 7
NUM_WORKERS = 0  # Avoid duplicating dataset arrays in worker processes.
REQUIRE_GPU = True
USE_TWO_GPUS = False  # Single RTX A6000.
SAVE_MODELS = True
SAVE_SPLIT_INDICES = True
SAVE_TEST_PREDICTIONS = True
SAVE_TEST_PROBABILITIES = True
SAVE_SHAP = False  # Install shap before enabling. Uses small bounded samples.
SHAP_BACKGROUND_ROWS = 50
SHAP_EXPLAIN_ROWS = 100

MODEL_CONFIGS = {
    'MLP_Embeddings': {'hidden_dims': [256, 128], 'dropout': .2,
                     'max_embedding_dim': 16, 'lr': .001, 'weight_decay': .00001, 'batch_size': 4096},
    'TabNet': {'n_d': 16, 'n_a': 16, 'n_steps': 3, 'gamma': 1.5,
               'cat_emb_dim': 8, 'lambda_sparse': .001, 'lr': .001,
               'weight_decay': .00001, 'batch_size': 4096, 'virtual_batch_size': 256},
    'FTTransformer': {'n_blocks': 2, 'd_block': 64, 'attention_n_heads': 8,
                      'attention_dropout': .2, 'ffn_dropout': .1, 'residual_dropout': 0.,
                      'lr': .0001, 'weight_decay': .00001, 'batch_size': 512},
}
DATETIME_COLS = ['Start_Time', 'End_Time', 'Weather_Timestamp']
BOOL_COLS = ['Amenity','Crossing','Give_Way','Junction','No_Exit','Railway','Station','Stop','Traffic_Signal']
SELECTED_FEATURES = ['Start_Lat', 'Start_Lng', 'Street', 'City', 'County', 'State', 'Zipcode', 'Timezone', 'Airport_Code', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Weather_Condition', 'Amenity', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Station', 'Stop', 'Traffic_Signal', 'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight', 'Year', 'Month', 'Hour', 'DayOfWeek', 'Season', 'Hour_sin', 'Hour_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Month_sin', 'Month_cos', 'Weather_Timestamp_diff_minutes']
CATEGORICAL_FEATURES = ['Street', 'City', 'County', 'State', 'Zipcode', 'Timezone', 'Airport_Code', 'Wind_Direction', 'Weather_Condition', 'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight', 'Season']
NUMERICAL_FEATURES = [c for c in SELECTED_FEATURES if c not in CATEGORICAL_FEATURES]
CSV_DTYPES = {c: "string" for c in CATEGORICAL_FEATURES}
assert "Distance(mi)" not in SELECTED_FEATURES
assert len(SELECTED_FEATURES) == 41
assert len(SEEDS) == 5 and len(set(SEEDS)) == 5
assert MIN_CATEGORY_COUNT >= 1 and MAX_CATEGORIES_PER_COLUMN >= 1
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT_DIR = OUTPUT_ROOT / f'dl_CW0_repeated_splits_{RUN_ID}'
OUT_DIR.mkdir(parents=True, exist_ok=False)
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if REQUIRE_GPU and DEVICE.type != 'cuda':
    raise RuntimeError('Select a Kaggle GPU accelerator, or set REQUIRE_GPU=False for a small CPU test.')
GPU_COUNT = torch.cuda.device_count()
print('Output:', OUT_DIR, '| Device:', DEVICE, '| GPUs:', GPU_COUNT)
if DEVICE.type == 'cuda':
    for i in range(GPU_COUNT):
        print(i, torch.cuda.get_device_name(i))
VERSIONS = {p: importlib.metadata.version(p) for p in [
    'numpy', 'pandas', 'scikit-learn', 'torch', 'pytorch-tabnet', 'rtdl_revisiting_models']}
(OUT_DIR / 'package_versions.json').write_text(json.dumps(VERSIONS, indent=2))

## Load the processed dataset and validate the selected features

In [ ]:
def engineer_features(df):
    """Validate the already engineered 41-predictor schema; exclude Distance(mi)."""
    required = [TARGET] + SELECTED_FEATURES
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Processed dataset is missing required columns: {missing}')
    df = df[required].copy()
    for c in CATEGORICAL_FEATURES:
        # Identifier values remain categories even when their CSV values look numeric.
        values = df[c].astype('string').str.strip()
        values = values.mask(values.eq(''), pd.NA)
        df[c] = values.fillna('Missing').astype(str)
    boolean_map = {'true': 1., 'false': 0., '1': 1., '0': 0.,
                   '1.0': 1., '0.0': 0., 'y': 1., 'n': 0., 'yes': 1., 'no': 0.}
    for c in BOOL_COLS:
        values = df[c].astype('string').str.strip().str.lower()
        values = values.mask(values.eq(''), pd.NA)
        converted = values.map(boolean_map)
        invalid = values.notna() & converted.isna()
        if invalid.any():
            raise ValueError(f'{c}: unsupported Boolean values {values[invalid].unique()[:5].tolist()}')
        df[c] = converted.astype('float32')
    for c in NUMERICAL_FEATURES:
        if c in BOOL_COLS:
            continue
        values = df[c]
        if not pd.api.types.is_numeric_dtype(values):
            values = values.astype('string').str.strip()
            values = values.mask(values.eq(''), pd.NA)
        converted = pd.to_numeric(values, errors='coerce')
        invalid = values.notna() & converted.isna()
        if invalid.any():
            raise ValueError(f'{c}: nonnumeric values in a numerical feature.')
        df[c] = converted.replace([np.inf, -np.inf], np.nan).astype('float32')
    # Missing feature values stay missing for training-fitted imputation.
    # Only missing targets are excluded; original row indices stay intact.
    df = df.dropna(subset=[TARGET])
    assert list(df.columns) == required
    return df


def save_feature_schema(frame):
    rows = []
    for c in SELECTED_FEATURES:
        values = frame[c]
        categorical = c in CATEGORICAL_FEATURES
        missing = values.eq('Missing') if categorical else values.isna()
        rows.append({'Feature': c,
                     'Role': 'categorical' if categorical else 'numerical',
                     'Dtype': str(values.dtype), 'Missing_count': int(missing.sum()),
                     'Missing_fraction': float(missing.mean()),
                     'Minimum': None if categorical else values.min(),
                     'Maximum': None if categorical else values.max()})
    pd.DataFrame(rows).to_csv(OUT_DIR / 'feature_schema.csv', index=False)
    (OUT_DIR / 'feature_schema.json').write_text(json.dumps({
        'target': TARGET, 'predictor_count': len(SELECTED_FEATURES),
        'feature_order': SELECTED_FEATURES, 'categorical': CATEGORICAL_FEATURES,
        'numerical': NUMERICAL_FEATURES}, indent=2))
    print(f'Feature schema: {len(SELECTED_FEATURES)} predictors | '
          f'{len(CATEGORICAL_FEATURES)} categorical | {len(NUMERICAL_FEATURES)} numerical', flush=True)

def load_full_data(path):
    if not path.is_file(): raise FileNotFoundError(f'Correct DATA_PATH: {path}')
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:
            members=[x for x in z.namelist() if x.lower().endswith('.csv') and not x.startswith('__MACOSX/')]
            if CSV_MEMBER is not None:
                if CSV_MEMBER not in members: raise ValueError(f'CSV_MEMBER not found. Available: {members}')
                selected=[CSV_MEMBER]
            elif len(members)==1: selected=members
            elif COMBINE_EXISTING_SPLITS:
                train=[x for x in members if 'train' in Path(x).name.lower()]
                test=[x for x in members if 'test' in Path(x).name.lower()]
                if len(train)!=1 or len(test)!=1 or train[0]==test[0]:
                    raise ValueError(f'Expected one train and one test member: {members}')
                selected=[train[0],test[0]]
            else: raise ValueError(f'ZIP has multiple CSVs: {members}. Set CSV_MEMBER or enable COMBINE_EXISTING_SPLITS.')
            frames=[]; identity=[]
            for member in selected:
                with z.open(member) as f: frame=pd.read_csv(f,low_memory=False,dtype=CSV_DTYPES)
                if TARGET not in frame: raise ValueError(f'{member} needs {TARGET}.')
                if frames and set(frame.columns)!=set(frames[0].columns): raise ValueError('CSV feature schemas differ.')
                frames.append(frame)
                info=z.getinfo(member)
                identity.append({'member':member,'crc32':f'{info.CRC:08x}','bytes':info.file_size,'rows':len(frame)})
            return pd.concat(frames,ignore_index=True),identity
    if path.suffix.lower() not in ['.csv','.gz']: raise ValueError('Use CSV, gzip CSV, or a ZIP of CSV files.')
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return pd.read_csv(path,low_memory=False,dtype=CSV_DTYPES),{'sha256':h.hexdigest(),'file':path.name}

raw, source_identity=load_full_data(DATA_PATH)
raw=raw.reset_index(drop=True)
original_rows=len(raw)
df=engineer_features(raw)
save_feature_schema(df)
del raw;gc.collect()
# Feature engineering retains original indices; labels missing from Severity are excluded.
row_ids=df.index.to_numpy(dtype=np.int64)
labels=pd.to_numeric(df[TARGET],errors='raise')
if not np.isfinite(labels).all() or not labels.isin([1,2,3,4]).all():
    raise ValueError('Severity must contain only 1,2,3,4 after removing missing labels.')
classes=sorted(labels.astype(int).unique().tolist())
missing_levels=sorted(set(REQUIRED_SEVERITY_LEVELS)-set(classes))
if missing_levels:
    raise ValueError(f'Full dataset is missing Severity levels {missing_levels}. All four levels are required; splitting cannot create missing classes.')
class_map={int(c):i for i,c in enumerate(classes)}
y=labels.astype(int).map(class_map).to_numpy(dtype=np.int64)
X=df.drop(columns=[TARGET]);del df;gc.collect()
n_classes=len(classes)
if n_classes<2: raise ValueError('At least two classes are required.')
counts=pd.Series(y).value_counts()
if counts.min()<10: raise ValueError('Not enough rows in a class for stable stratified 64/16/20 splitting.')
print('Full pool:',len(X),'| excluded missing-label rows:',original_rows-len(X),'| classes:',class_map)
protocol = {
    'method': 'repeated stratified holdouts', 'seeds': SEEDS,
    'fractions': {'train': .64, 'validation': .16, 'test': .20},
    'source': source_identity, 'original_rows': original_rows, 'usable_rows': len(X),
    'class_map': class_map, 'selected_features': list(X.columns), 'models': MODEL_NAMES,
    'class_weight_cap': CLASS_WEIGHT_CAP, 'threshold_beta': THRESHOLD_BETA,
    'threshold_rule': 'independent OvR F_beta; highest probability passing class; training-majority fallback',
    'early_stopping_metric': 'validation macro-F1', 'max_epochs': MAX_EPOCHS,
    'patience': PATIENCE, 'device': str(DEVICE), 'use_two_gpus': USE_TWO_GPUS,
    'category_min_count': MIN_CATEGORY_COUNT, 'category_max_count': MAX_CATEGORIES_PER_COLUMN,
    'model_configs': MODEL_CONFIGS, 'std_ddof': 1,
}
protocol['class_weight_mode'] = CLASS_WEIGHT_MODE
protocol['use_class_weight'] = USE_CLASS_WEIGHT
(OUT_DIR / 'protocol.json').write_text(json.dumps(protocol, indent=2))
if ML_RESULTS_DIR is not None:
    reference = json.loads((Path(ML_RESULTS_DIR) / 'protocol.json').read_text())
    for key in ['source', 'original_rows', 'usable_rows', 'selected_features']:
        if reference[key] != protocol[key]:
            raise ValueError(f'ML and DL differ in {key}; use the same full input and feature pipeline.')

## Training-only preprocessing and unit weights

No target encoding is used for DL. Category IDs feed learned embedding layers.

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def fit_preprocessor(frame):
    numerical = [c for c in NUMERICAL_FEATURES if c in frame]
    categorical = [c for c in CATEGORICAL_FEATURES if c in frame]
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    scaler = StandardScaler()
    if numerical:
        imputed = imputer.fit_transform(frame[numerical])
        scaler.fit(imputed)
    mappings = {}
    for c in categorical:
        values = frame[c].astype('string').fillna('Missing')
        counts = values[values != 'Missing'].value_counts()
        retained = counts[counts >= MIN_CATEGORY_COUNT].head(MAX_CATEGORIES_PER_COLUMN).index
        # Include rare training categories with ID 2; unseen validation/test values receive 0.
        mapping = {str(value): 2 for value in counts.index}
        mapping.update({str(value): i + 3 for i, value in enumerate(retained)})
        mapping['Missing'] = 1
        mappings[c] = mapping
    return {'numerical': numerical, 'categorical': categorical, 'mappings': mappings,
            'imputer': imputer, 'scaler': scaler, 'input_columns': list(frame.columns),
            'cat_sizes': [max(3, max(mappings[c].values()) + 1) for c in categorical]}


def transform_dl(frame, bundle):
    if bundle['numerical']:
        numerical = bundle['scaler'].transform(bundle['imputer'].transform(frame[bundle['numerical']]))
        numerical = np.asarray(numerical, dtype='float32')
    else:
        numerical = np.empty((len(frame), 0), dtype='float32')
    categorical = np.empty((len(frame), len(bundle['categorical'])), dtype='int32')
    for i, c in enumerate(bundle['categorical']):
        categorical[:, i] = frame[c].astype('string').fillna('Missing').map(
            bundle['mappings'][c]).fillna(0).to_numpy(dtype='int32')
    assert np.isfinite(numerical).all()
    # Embedding IDs stay unscaled and must fit each train-fitted vocabulary.
    for i, size in enumerate(bundle['cat_sizes']):
        if len(categorical) and (categorical[:, i].min() < 0 or categorical[:, i].max() >= size):
            raise ValueError(f'Out-of-range embedding ID for {bundle["categorical"][i]}.')
    return numerical, categorical


def get_class_weights(labels):
    weights = np.ones(n_classes)
    if USE_CLASS_WEIGHT:
        weights = compute_class_weight('balanced', classes=np.arange(n_classes), y=labels)
        if CLASS_WEIGHT_CAP is not None:
            weights = np.minimum(weights, weights.min() * CLASS_WEIGHT_CAP)
        weights /= weights[labels].mean()
    return weights.astype('float32')


class AccidentDataset(Dataset):
    def __init__(self, numerical, categorical, labels=None):
        self.numerical = numerical
        self.categorical = categorical
        self.labels = labels

    def __len__(self):
        return len(self.numerical)

    def __getitem__(self, i):
        values = (torch.from_numpy(self.numerical[i]),
                  torch.from_numpy(self.categorical[i]).long())
        return values if self.labels is None else (*values, int(self.labels[i]))


def make_loader(numerical, categorical, labels, batch_size, shuffle=False, seed=42):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(AccidentDataset(numerical, categorical, labels), batch_size=batch_size,
        shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda',
        generator=generator)

## Architectures and weighted training

All embeddings are model parameters updated by the training optimizer.

In [ ]:
class EmbeddingMLP(nn.Module):
    def __init__(self, n_numerical, cat_sizes, config):
        super().__init__()
        dims = [min(config['max_embedding_dim'], max(2, int(np.ceil(np.sqrt(size))))) for size in cat_sizes]
        self.embeddings = nn.ModuleList([nn.Embedding(size, dim) for size, dim in zip(cat_sizes, dims)])
        width = n_numerical + sum(dims)
        layers = []
        for hidden in config['hidden_dims']:
            layers += [nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(config['dropout'])]
            width = hidden
        layers.append(nn.Linear(width, n_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, numerical, categorical):
        embedded = [layer(categorical[:, i]) for i, layer in enumerate(self.embeddings)]
        return self.network(torch.cat([numerical] + embedded, dim=1))


class TransformerInputs(nn.Module):
    def __init__(self, n_numerical, cat_sizes, config):
        super().__init__()
        self.has_numerical = n_numerical > 0
        self.has_categorical = len(cat_sizes) > 0
        self.model = FTTransformer(
            n_cont_features=n_numerical, cat_cardinalities=cat_sizes, d_out=n_classes,
            n_blocks=config['n_blocks'], d_block=config['d_block'],
            attention_n_heads=config['attention_n_heads'], attention_dropout=config['attention_dropout'],
            ffn_d_hidden=None, ffn_d_hidden_multiplier=4/3, ffn_dropout=config['ffn_dropout'],
            residual_dropout=config['residual_dropout'])

    def forward(self, numerical, categorical):
        return self.model(numerical if self.has_numerical else None,
                          categorical if self.has_categorical else None)


def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def create_torch_model(name, bundle):
    config = MODEL_CONFIGS[name]
    n_num = len(bundle['numerical'])
    model = EmbeddingMLP(n_num, bundle['cat_sizes'], config) if name == 'MLP_Embeddings' else             TransformerInputs(n_num, bundle['cat_sizes'], config)
    if bundle['cat_sizes'] and not any(isinstance(layer, nn.Embedding) for layer in model.modules()):
        raise RuntimeError(f'{name}: categorical embedding layers are missing.')
    model = model.to(DEVICE)
    # Use the documented FT-Transformer parameter groups for weight decay.
    groups = model.model.make_parameter_groups() if name == 'FTTransformer' else model.parameters()
    optimizer = torch.optim.AdamW(groups, lr=config['lr'], weight_decay=config['weight_decay'])
    if USE_TWO_GPUS and GPU_COUNT >= 2:
        model = nn.DataParallel(model, device_ids=[0, 1])
    return model, optimizer


@torch.inference_mode()
def evaluate_torch(model, numerical, categorical, labels, batch_size):
    model.eval()
    probabilities = np.empty((len(labels), n_classes), dtype='float32')
    total_loss, offset = 0., 0
    for num, cat, target in make_loader(numerical, categorical, labels, batch_size):
        num, cat, target = num.to(DEVICE), cat.to(DEVICE), target.to(DEVICE)
        logits = model(num, cat)
        total_loss += nn.functional.cross_entropy(logits, target, reduction='sum').item()
        p = logits.softmax(dim=1).cpu().numpy()
        probabilities[offset:offset + len(p)] = p
        offset += len(p)
    return total_loss / len(labels), f1_score(labels, probabilities.argmax(1), average='macro', zero_division=0)


@torch.inference_mode()
def predict_torch(model, numerical, categorical, batch_size):
    model.eval()
    output = np.empty((len(numerical), n_classes), dtype='float32')
    offset = 0
    for num, cat in make_loader(numerical, categorical, None, batch_size):
        p = model(num.to(DEVICE), cat.to(DEVICE)).softmax(dim=1).cpu().numpy()
        output[offset:offset + len(p)] = p
        offset += len(p)
    return output


def train_torch(name, train_data, val_data, y_train, y_val, bundle, weights, seed, run_dir):
    model, optimizer = create_torch_model(name, bundle)
    batch_size = MODEL_CONFIGS[name]['batch_size']
    loader = make_loader(*train_data, y_train, batch_size, shuffle=True, seed=seed)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, device=DEVICE), reduction='sum')
    best_score, stale = -np.inf, 0
    checkpoint = run_dir / f'{name}_best.pt'
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        weighted_sum, weight_sum = 0., 0.
        for num, cat, target in loader:
            num, cat, target = num.to(DEVICE), cat.to(DEVICE), target.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(num, cat)
            numerator = criterion(logits, target)
            denominator = criterion.weight[target].sum()
            loss = numerator / denominator
            if not torch.isfinite(loss):
                raise RuntimeError(f'{name}: nonfinite training loss.')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.)
            optimizer.step()
            weighted_sum += numerator.item()
            weight_sum += denominator.item()
        val_loss, val_f1 = evaluate_torch(model, *val_data, y_val, batch_size)
        history.append({'Epoch': epoch, 'Train_Weighted_Loss': weighted_sum / weight_sum,
                        'Validation_LogLoss': val_loss, 'Validation_Macro_F1': val_f1})
        pd.DataFrame(history).to_csv(run_dir / f'{name}_training_history.csv', index=False)
        print(f'{name} epoch {epoch}: train loss {weighted_sum/weight_sum:.4f}, val macro-F1 {val_f1:.4f}', flush=True)
        if val_f1 > best_score + 1e-6:
            best_score, stale = val_f1, 0
            torch.save(unwrap(model).state_dict(), checkpoint)
        else:
            stale += 1
        if stale >= PATIENCE:
            break
    unwrap(model).load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
    if not SAVE_MODELS:
        checkpoint.unlink()
    return model, pd.DataFrame(history)


class MacroF1(Metric):
    def __init__(self):
        self._name = 'macro_f1'
        self._maximize = True

    def __call__(self, y_true, y_score):
        return f1_score(y_true, y_score.argmax(axis=1), average='macro', zero_division=0)


def combine_tabnet(data):
    return np.concatenate(data, axis=1).astype('float32')


def train_tabnet(train_data, val_data, y_train, y_val, bundle, weights, seed, run_dir):
    config = MODEL_CONFIGS['TabNet']
    train_x, val_x = combine_tabnet(train_data), combine_tabnet(val_data)
    categorical_positions = list(range(len(bundle['numerical']), train_x.shape[1]))
    model = TabNetClassifier(n_d=config['n_d'], n_a=config['n_a'], n_steps=config['n_steps'],
        gamma=config['gamma'], cat_idxs=categorical_positions, cat_dims=bundle['cat_sizes'],
        cat_emb_dim=config['cat_emb_dim'], lambda_sparse=config['lambda_sparse'],
        optimizer_fn=torch.optim.AdamW,
        optimizer_params={'lr': config['lr'], 'weight_decay': config['weight_decay']},
        seed=seed, device_name=str(DEVICE), verbose=1)
    loss = nn.CrossEntropyLoss(weight=torch.tensor(weights, device=DEVICE))
    # weights=0 disables the library's weighted sampling. Weighting is in the loss only.
    batch_size = min(config['batch_size'], len(y_train))
    virtual_batch = min(config['virtual_batch_size'], batch_size)
    model.fit(train_x, y_train, eval_set=[(val_x, y_val)], eval_name=['validation'],
        eval_metric=['logloss', MacroF1], loss_fn=loss, weights=0,
        max_epochs=MAX_EPOCHS, patience=PATIENCE, batch_size=batch_size,
        virtual_batch_size=virtual_batch, num_workers=NUM_WORKERS,
        drop_last=len(y_train) % batch_size == 1)
    history = pd.DataFrame({
        'Epoch': np.arange(1, len(model.history['loss']) + 1),
        'Train_Weighted_Loss': model.history['loss'],
        'Validation_LogLoss': model.history['validation_logloss'],
        'Validation_Macro_F1': model.history['validation_macro_f1'],
    })
    history.to_csv(run_dir / 'TabNet_training_history.csv', index=False)
    if SAVE_MODELS:
        model.save_model(str(run_dir / 'TabNet_best'))
    del train_x, val_x
    gc.collect()
    return model, history


def predict_dl(name, model, data):
    if name == 'TabNet':
        result = np.empty((len(data[0]), n_classes), dtype='float32')
        for start in range(0, len(result), 100_000):
            end = min(start + 100_000, len(result))
            result[start:end] = model.predict_proba(combine_tabnet((data[0][start:end], data[1][start:end])))
    else:
        result = predict_torch(model, *data, MODEL_CONFIGS[name]['batch_size'])
    if not np.isfinite(result).all() or not np.allclose(result.sum(1), 1, atol=1e-4):
        raise RuntimeError('Invalid probabilities.')
    return result


def save_history_plot(history, name, run_dir):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history.Epoch, history.Train_Weighted_Loss, label='Training weighted loss')
    axes[0].plot(history.Epoch, history.Validation_LogLoss, label='Validation unweighted log loss')
    axes[0].set(xlabel='Epoch', ylabel='Loss', title=name)
    axes[0].legend()
    axes[1].plot(history.Epoch, history.Validation_Macro_F1)
    axes[1].set(xlabel='Epoch', ylabel='Validation macro-F1', ylim=(0, 1))
    fig.tight_layout(); fig.savefig(run_dir / f'{name}_training_curves.png', dpi=180)
    plt.close(fig)

In [ ]:
def calc_metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Macro_Precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Macro_Recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'Macro_F1': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }


def tune_thresholds(y_val, probabilities):
    """Preserve the original independent one-vs-rest validation F_beta objective."""
    thresholds = []
    for c in range(n_classes):
        precision, recall, candidates = precision_recall_curve(y_val == c, probabilities[:, c])
        beta2 = THRESHOLD_BETA ** 2
        denominator = beta2 * precision[:-1] + recall[:-1]
        scores = np.divide((1 + beta2) * precision[:-1] * recall[:-1], denominator,
                           out=np.zeros_like(denominator), where=denominator > 0)
        thresholds.append(float(candidates[np.argmax(scores)]))
    return np.asarray(thresholds)


def predict_with_thresholds(probabilities, thresholds, fallback_class):
    passed = probabilities >= thresholds[None, :]
    predictions = np.where(passed, probabilities, -np.inf).argmax(axis=1)
    predictions[~passed.any(axis=1)] = fallback_class
    return predictions

## Optional SHAP for trained probability outputs

Install `shap` before enabling `SAVE_SHAP`. No test explanations are used to select models or thresholds.

In [ ]:
def save_shap_results(name, model, train_data, test_data, bundle, seed, run_dir, test_row_ids):
    if not SAVE_SHAP:
        return
    import shap
    rng = np.random.default_rng(seed)
    train_positions = rng.choice(len(train_data[0]), min(SHAP_BACKGROUND_ROWS, len(train_data[0])), replace=False)
    test_positions = rng.choice(len(test_data[0]), min(SHAP_EXPLAIN_ROWS, len(test_data[0])), replace=False)
    background = combine_tabnet((train_data[0][train_positions], train_data[1][train_positions]))
    samples = combine_tabnet((test_data[0][test_positions], test_data[1][test_positions]))
    n_num = len(bundle['numerical'])
    names = bundle['numerical'] + bundle['categorical']
    def probability_predictor(array):
        return predict_dl(name, model, (array[:, :n_num].astype('float32'), array[:, n_num:].astype('int32')))
    explainer = shap.Explainer(probability_predictor, shap.maskers.Independent(background, max_samples=len(background)),
                             algorithm='permutation', feature_names=names, seed=seed)
    explanation = explainer(samples, max_evals=2 * len(names) + 1)
    np.savez_compressed(run_dir / f'{name}_shap_values.npz', values=explanation.values,
        base_values=explanation.base_values, row_ids=test_row_ids[test_positions],
        feature_names=np.asarray(names, dtype=str), classes=np.asarray(classes))
    pd.DataFrame(np.abs(explanation.values).mean(axis=0), index=names,
                 columns=[f'Severity_{c}' for c in classes]).to_csv(run_dir / f'{name}_shap_mean_absolute.csv')

## Five seeds: training and paired test evaluation

In [ ]:
def split_for_seed(y, seed):
    """Stratified 64/16/20 split; reject any partition missing Severity 1–4."""
    y = np.asarray(y)
    expected = set(range(len(REQUIRED_SEVERITY_LEVELS)))
    if set(np.unique(y)) != expected:
        raise ValueError('Encoded labels must include all four Severity levels (1, 2, 3, 4).')
    positions = np.arange(len(y), dtype=np.int64)
    dev, test = train_test_split(positions, test_size=.20, stratify=y, random_state=seed)
    train, val = train_test_split(dev, test_size=.20, stratify=y[dev], random_state=seed)
    combined = np.concatenate([train, val, test])
    if len(combined) != len(y) or len(np.unique(combined)) != len(y):
        raise RuntimeError(f'Seed {seed}: partitions overlap or do not cover the full dataset.')
    for name, idx in [('Train', train), ('Validation', val), ('Test', test)]:
        missing = expected - set(np.unique(y[idx]))
        if missing:
            levels = [REQUIRED_SEVERITY_LEVELS[c] for c in sorted(missing)]
            raise ValueError(f'Seed {seed}: {name} is missing Severity {levels}. '
                             'Add more records for the rare classes before running training.')
    return train, val, test

def severity_counts_for_split(y, train_idx, val_idx, test_idx):
    """Counts use original Severity labels, not the encoded 0–3 labels."""
    table = pd.DataFrame({
        name: np.bincount(y[idx], minlength=len(REQUIRED_SEVERITY_LEVELS))
        for name, idx in [('Train', train_idx), ('Validation', val_idx), ('Test', test_idx)]
    }, index=pd.Index(REQUIRED_SEVERITY_LEVELS, name='Severity'))
    if not (table.to_numpy() > 0).all():
        raise ValueError('Every partition must contain at least one record from each Severity level.')
    return table

# Validate every planned seed before any expensive model training begins.
for seed in SEEDS:
    split_for_seed(y, seed)
print('Preflight passed: every seed includes all four severity levels in every partition.')


def report_predictions(model_name, stage, y_true, prediction, run_dir, seed, run_number):
    matrix = confusion_matrix(y_true, prediction, labels=range(n_classes))
    pd.DataFrame(matrix, index=classes, columns=classes).rename_axis('Actual Severity').to_csv(
        run_dir / f'{model_name}_{stage}_confusion_matrix.csv')
    for normalized in (False, True):
        values = matrix / matrix.sum(axis=1, keepdims=True) if normalized else matrix
        fig, ax = plt.subplots(figsize=(5, 4))
        im = ax.imshow(values, cmap='Blues', vmin=0, vmax=1 if normalized else None)
        for i in range(n_classes):
            for j in range(n_classes):
                ax.text(j, i, f'{values[i,j]:.2f}' if normalized else f'{values[i,j]:,}',
                        ha='center', va='center', fontsize=8,
                        color='white' if values[i,j] > values.max()/2 else 'black')
        ax.set(xticks=range(n_classes), yticks=range(n_classes), xticklabels=classes,
               yticklabels=classes, xlabel='Predicted Severity', ylabel='Actual Severity',
               title=f'{model_name} — {stage}')
        fig.colorbar(im, ax=ax); fig.tight_layout()
        suffix = '_normalized' if normalized else ''
        fig.savefig(run_dir / f'{model_name}_{stage}_confusion_matrix{suffix}.png', dpi=160)
        plt.close(fig)
    report = pd.DataFrame(classification_report(y_true, prediction, labels=range(n_classes),
        target_names=[str(c) for c in classes], output_dict=True, zero_division=0)).T
    report = report.drop(index='weighted avg', errors='ignore')
    # Binary one-vs-rest accuracy includes TP and TN for each encoded class.
    for encoded_class, label in enumerate(classes):
        report.loc[str(label), 'Accuracy'] = accuracy_score(
            np.asarray(y_true) == encoded_class, np.asarray(prediction) == encoded_class)
    report.to_csv(run_dir / f'{model_name}_{stage}_classification_report.csv')
    for label in classes:
        rec = report.loc[str(label)]
        per_class_rows.append({'Run': run_number, 'Seed': seed, 'Model': model_name,
            'Stage': stage, 'Severity': label, 'Accuracy': rec['Accuracy'],
            'Precision': rec['precision'],
            'Recall': rec['recall'], 'F1': rec['f1-score'], 'Support': int(rec['support'])})


def flush_results():
    pd.DataFrame(metric_rows).to_csv(OUT_DIR / 'run_test_metrics.csv', index=False)
    pd.DataFrame(delta_rows).to_csv(OUT_DIR / 'run_threshold_deltas.csv', index=False)
    pd.DataFrame(per_class_rows).to_csv(OUT_DIR / 'run_per_class_metrics.csv', index=False)

In [ ]:
metric_rows, delta_rows, per_class_rows = [], [], []
for run_number, seed in enumerate(SEEDS, 1):
    run_dir = OUT_DIR / f'run_{run_number}_seed_{seed}'
    run_dir.mkdir()
    train_idx, val_idx, test_idx = split_for_seed(y, seed)
    counts = severity_counts_for_split(y, train_idx, val_idx, test_idx)
    display(counts)
    counts.to_csv(run_dir / 'split_severity_counts.csv')
    manifest = {'seed': seed, 'run': run_number, 'counts': {}, 'index_sha256': {}, 'class_counts': {}}
    for part, idx in [('train', train_idx), ('validation', val_idx), ('test', test_idx)]:
        manifest['counts'][part] = len(idx)
        manifest['index_sha256'][part] = hashlib.sha256(np.sort(row_ids[idx]).tobytes()).hexdigest()
        manifest['class_counts'][part] = {str(classes[c]): int((y[idx] == c).sum()) for c in range(n_classes)}
    if ML_RESULTS_DIR is not None:
        ml_manifests = list(Path(ML_RESULTS_DIR).glob(f'run_*_seed_{seed}/split_manifest.json'))
        if len(ml_manifests) != 1:
            raise ValueError(f'Expected one ML split manifest for seed {seed}.')
        reference = json.loads(ml_manifests[0].read_text())
        if manifest['index_sha256'] != reference['index_sha256']:
            raise ValueError(f'Seed {seed}: ML and DL split rows differ.')
    (run_dir / 'split_manifest.json').write_text(json.dumps(manifest, indent=2))
    if SAVE_SPLIT_INDICES:
        np.savez_compressed(run_dir / 'split_row_ids.npz', train=row_ids[train_idx],
                            validation=row_ids[val_idx], test=row_ids[test_idx])
    bundle = fit_preprocessor(X.iloc[train_idx])
    joblib.dump(bundle, run_dir / 'preprocessing.pkl')
    pd.DataFrame({'Feature': bundle['categorical'], 'Category_ID_Count': bundle['cat_sizes']}).to_csv(
        run_dir / 'category_sizes.csv', index=False)
    train_data = transform_dl(X.iloc[train_idx], bundle)
    val_data = transform_dl(X.iloc[val_idx], bundle)
    test_data = transform_dl(X.iloc[test_idx], bundle)
    y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
    weights = get_class_weights(y_train)
    (run_dir / 'class_weights.json').write_text(json.dumps(dict(zip(map(str, classes), weights.tolist())), indent=2))
    majority = int(np.bincount(y_train).argmax())
    for name in MODEL_NAMES:
        seed_everything(seed)
        print(f'Run {run_number}/{len(SEEDS)} | seed {seed} | {name}', flush=True)
        (run_dir / 'run_status.json').write_text(json.dumps({'status': 'training', 'model': name}))
        (run_dir / f'{name}_parameters.json').write_text(json.dumps(MODEL_CONFIGS[name], indent=2))
        started = time.perf_counter()
        try:
            if name == 'TabNet':
                model, history = train_tabnet(train_data, val_data, y_train, y_val, bundle, weights, seed, run_dir)
            else:
                model, history = train_torch(name, train_data, val_data, y_train, y_val, bundle, weights, seed, run_dir)
            fit_seconds = time.perf_counter() - started
            save_history_plot(history, name, run_dir)
            thresholds = tune_thresholds(y_val, predict_dl(name, model, val_data))
            (run_dir / f'{name}_thresholds.json').write_text(json.dumps({
                'thresholds': dict(zip(map(str, classes), thresholds.tolist())),
                'fallback_severity': classes[majority], 'beta': THRESHOLD_BETA,
                'objective': 'independent one-vs-rest F_beta',
                'rule': 'highest original probability among passing classes; otherwise training majority'}, indent=2))
            probabilities = predict_dl(name, model, test_data)
            before = probabilities.argmax(1)
            after = predict_with_thresholds(probabilities, thresholds, majority)
            scores = {}
            for stage, prediction in [('Before', before), ('After', after)]:
                scores[stage] = calc_metrics(y_test, prediction)
                metric_rows.append({'Run': run_number, 'Seed': seed, 'Model': name, 'Stage': stage,
                                    'Fit_seconds': fit_seconds, **scores[stage]})
                report_predictions(name, stage, y_test, prediction, run_dir, seed, run_number)
            delta_rows.append({'Run': run_number, 'Seed': seed, 'Model': name,
                **{m: scores['After'][m] - scores['Before'][m] for m in METRICS}})
            if SAVE_TEST_PREDICTIONS:
                pd.DataFrame({'Row_ID': row_ids[test_idx], 'y_true': np.asarray(classes)[y_test],
                    'Before': np.asarray(classes)[before], 'After': np.asarray(classes)[after]}).to_csv(
                        run_dir / f'{name}_test_predictions.csv', index=False)
            if SAVE_TEST_PROBABILITIES:
                np.savez_compressed(run_dir / f'{name}_test_probabilities.npz', row_ids=row_ids[test_idx],
                    classes=np.asarray(classes), y_true=np.asarray(classes)[y_test], probabilities=probabilities)
            save_shap_results(name, model, train_data, test_data, bundle, seed, run_dir, row_ids[test_idx])
            if name == 'TabNet':
                pd.DataFrame({'Feature': bundle['numerical'] + bundle['categorical'],
                              'Importance': model.feature_importances_}).sort_values('Importance', ascending=False).to_csv(
                                  run_dir / 'TabNet_feature_importance.csv', index=False)
            print(name, 'Before:', scores['Before'], 'After:', scores['After'], flush=True)
            flush_results()
        except Exception as exc:
            (run_dir / 'run_status.json').write_text(json.dumps({'status': 'failed', 'model': name, 'error': str(exc)}))
            raise
        del model, history, probabilities, before, after
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    (run_dir / 'run_status.json').write_text(json.dumps({'status': 'complete', 'seed': seed}))
    del train_data, val_data, test_data, bundle, weights
    gc.collect()

assert len(metric_rows) == 2 * len(SEEDS) * len(MODEL_NAMES)
assert len(delta_rows) == len(SEEDS) * len(MODEL_NAMES)
print('All requested DL runs completed.')

## Mean ± sample standard deviation and result graphs

Paired threshold changes use after minus before for each model/run. Test sets overlap across runs;
sample SD is descriptive variability, not a confidence interval. Learning curves compare weighted training loss to unweighted validation
log loss; their numerical levels are not directly comparable. TabNet training loss includes sparsity regularization.

In [ ]:
metrics_df=pd.DataFrame(metric_rows);deltas_df=pd.DataFrame(delta_rows);per_class_df=pd.DataFrame(per_class_rows)

def save_mean_std(frame,groupby,metrics,stem):
    summary=frame.groupby(groupby)[metrics].agg(['mean','std'])
    flat=summary.copy();flat.columns=[f'{m}_{stat}' for m,stat in flat.columns]
    flat.to_csv(OUT_DIR/f'{stem}_numeric.csv')
    formatted=pd.DataFrame(index=summary.index)
    for m in metrics:
        formatted[m]=[f'{a:.4f} ± {b:.4f}' for a,b in zip(summary[(m,'mean')],summary[(m,'std')])]
    formatted.to_csv(OUT_DIR/f'{stem}_scores.csv')
    display(formatted)
    return summary

print('Five repeated splits: test scores before threshold tuning')
before_summary=save_mean_std(metrics_df[metrics_df.Stage=='Before'],'Model',METRICS,'before_test')
print('Five repeated splits: test scores after threshold tuning')
after_summary=save_mean_std(metrics_df[metrics_df.Stage=='After'],'Model',METRICS,'after_test')
print('Five repeated splits: threshold impact (after − before)')
delta_summary=save_mean_std(deltas_df,'Model',METRICS,'threshold_impact')
class_summary=save_mean_std(per_class_df,['Model','Stage','Severity'],['Accuracy','Precision','Recall','F1'],'per_class')
class_delta=per_class_df.pivot(index=['Run','Seed','Model','Severity'],columns='Stage',values=['Accuracy','Precision','Recall','F1'])
class_delta=pd.DataFrame({m:class_delta[(m,'After')]-class_delta[(m,'Before')] for m in ['Accuracy','Precision','Recall','F1']}).reset_index()
class_delta.to_csv(OUT_DIR/'run_per_class_deltas.csv',index=False)
class_delta_summary=save_mean_std(class_delta,['Model','Severity'],['Accuracy','Precision','Recall','F1'],'per_class_impact')

x=np.arange(len(MODEL_NAMES));width=.36
fig,axes=plt.subplots(2,2,figsize=(14,9))
for ax,m in zip(axes.flat,METRICS):
    for offset,summary,label,color in [(-width/2,before_summary,'Before','#4477AA'),(width/2,after_summary,'After','#EE6677')]:
        table=summary.reindex(MODEL_NAMES)
        ax.bar(x+offset,table[(m,'mean')],width,yerr=table[(m,'std')].fillna(0),capsize=4,label=label,color=color)
    ax.set_xticks(x,MODEL_NAMES,rotation=25,ha='right');ax.set_ylim(0,1);ax.set_ylabel('Test score');ax.set_title(m.replace('_',' '));ax.legend();ax.grid(axis='y',alpha=.2)
fig.suptitle('Five repeated splits: test mean ± sample SD before and after threshold tuning')
fig.tight_layout();fig.savefig(OUT_DIR/'before_after_scores.png',dpi=300);fig.savefig(OUT_DIR/'before_after_scores.pdf');plt.show()
fig,axes=plt.subplots(2,2,figsize=(14,9))
for ax,m in zip(axes.flat,METRICS):
    table=delta_summary.reindex(MODEL_NAMES)
    ax.bar(x,table[(m,'mean')],yerr=table[(m,'std')].fillna(0),capsize=4,color='#228833')
    ax.axhline(0,color='black',linewidth=.8);ax.set_xticks(x,MODEL_NAMES,rotation=25,ha='right')
    ax.set_title(m.replace('_',' '));ax.set_ylabel('After − before');ax.grid(axis='y',alpha=.2)
fig.suptitle('Five repeated splits: threshold tuning impact (mean ± sample SD)')
fig.tight_layout();fig.savefig(OUT_DIR/'threshold_impact_scores.png',dpi=300);plt.show()
fig,axes=plt.subplots(1,len(MODEL_NAMES),figsize=(5*len(MODEL_NAMES),4.8),sharey=True)
for ax,name in zip(axes,MODEL_NAMES):
    for stage,color in [('Before','#4477AA'),('After','#EE6677')]:
        rows=metrics_df[(metrics_df.Model==name)&(metrics_df.Stage==stage)].sort_values('Run')
        ax.plot(rows.Run,rows.Macro_F1,'o-',label=stage,color=color)
    ax.set_title(name);ax.set_xticks(range(1,len(SEEDS)+1));ax.set_xlabel('Run');ax.set_ylim(0,1);ax.legend();ax.grid(alpha=.2)
axes[0].set_ylabel('Test Macro-F1');fig.tight_layout();fig.savefig(OUT_DIR/'macro_f1_by_run.png',dpi=300);plt.show()
fig,axes=plt.subplots(1,len(MODEL_NAMES),figsize=(5*len(MODEL_NAMES),5),sharey=True)
for ax,name in zip(axes,MODEL_NAMES):
    for offset,stage,color in [(-width/2,'Before','#4477AA'),(width/2,'After','#EE6677')]:
        part=class_summary.loc[(name,stage)].reindex(classes)
        ax.bar(np.arange(len(classes))+offset,part[('F1','mean')],width,yerr=part[('F1','std')].fillna(0),capsize=3,label=stage,color=color)
    ax.set_title(name);ax.set_xticks(np.arange(len(classes)),classes);ax.set_xlabel('Severity');ax.set_ylim(0,1);ax.legend()
axes[0].set_ylabel('Per-class test F1');fig.suptitle('Five repeated splits: per-severity test F1 (mean ± sample SD)')
fig.tight_layout();fig.savefig(OUT_DIR/'per_severity_f1.png',dpi=300);plt.show()
(OUT_DIR/'run_status.txt').write_text('SUCCESS: all requested seeds and three DL models completed.\n')
archive=shutil.make_archive(str(OUT_DIR),'zip',OUT_DIR)
print('Download results:',archive);display(FileLink(archive))

## Reload trained models

Retain this notebook and matching package versions. Reload preprocessing with `joblib.load`, apply
`engineer_features` to new raw rows, and call `transform_dl`. For MLP/FT-Transformer recreate the model
with saved `protocol.json` settings and the bundle's category sizes, then load `*_best.pt` into the
unwrapped model. For TabNet use `TabNetClassifier().load_model(path_to_TabNet_best_zip)`.
Apply the saved threshold vector in Severity 1–4 order and saved fallback severity (convert it to 0–3
internally). No fitting, category remapping or gradient updates should occur during prediction.

Method references: [TabNet implementation](https://github.com/dreamquark-ai/tabnet) and
[FT-Transformer official package](https://github.com/yandex-research/rtdl-revisiting-models/tree/main/package).